### Install Libraries

In [ ]:
# !pip install torch transformers datasets peft trl accelerate bitsandbytes

### Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig
import torch

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

### Load Dataset

In [ ]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

### Setting peft and Qlora to fit on GPU

In [ ]:
# QLoRA quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# ── Step 3: LoRA config ──
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

### Load and Preprocess Dataset

In [ ]:
# Set chat template for base model
tokenizer.chat_template = (
    "{{ bos_token }}"                        # <s> at the very start
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "[INST] {{ message['content'] }} [/INST]"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] }}{{ eos_token }}" # </s> after every assistant turn
    "{% endif %}"
    "{% endfor %}"
)

# ── Step 2: Load and format dataset ──
# raw_dataset = load_dataset(
#     "HuggingFaceH4/ultrachat_200k",
#     split="train_sft"
# )

def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return example

dataset = dataset.map(
    apply_chat_template,
    num_proc=1,
    desc="Applying chat template",
    # remove all other columns — leave only "text"
    # this prevents TRL from seeing "prompt" column and going into assistant_only_loss path
    remove_columns=dataset.column_names,
)

# train_dataset = dataset.select(range(200))
# Verify — should ONLY have "text" column now
print("Columns:", dataset.column_names)   # → ['text']
print("Sample:\n", dataset[0]["text"][:200])

### Setting SFT config and trainer

In [ ]:

# ── Step 4: Training args ──
training_args = SFTConfig(
    output_dir="./sft-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    # max_seq_length=2048,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=200,
    dataset_text_field="text",
)

In [ ]:
# ── Step 5: Trainer ──
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)


### to start training run command below

In [ ]:
trainer.train()

### Saved trained model

In [ ]:
# ── Step 6: Save ──
trainer.model.save_pretrained("./sft-lora-adapters")
tokenizer.save_pretrained("./sft-lora-adapters")
print("Saved to ./sft-lora-adapters")

### Below Step is to Load trained model and predict based on query

In [ ]:
# ============================================================
# LOAD OPTION A — keep adapters separate (less RAM, can swap)
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./sft-lora-adapters")
model.eval()

tokenizer = AutoTokenizer.from_pretrained("./sft-lora-adapters")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"    # left padding for generation

In [ ]:
# ============================================================
# LOAD OPTION B — merge adapters into base (faster inference)
# ============================================================
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./sft-lora-adapters")
model = model.merge_and_unload()   # fuses LoRA weights into base permanently
model.eval()

# save merged so you don't need to merge again next time
model.save_pretrained("./sft-lora-merged")

# ── Load from merged (fastest — no PEFT overhead at all) ──
model = AutoModelForCausalLM.from_pretrained(
    "./sft-lora-merged",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

In [ ]:
# ============================================================
# PREDICT — identical interface regardless of load option
# ============================================================
import torch

def predict(prompt: str, max_new_tokens: int = 200) -> str:
    # Format using the same template used during training
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # adds [/INST] at end — model generates after this
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    # Fix: explicitly send to cuda instead of model.device
    # model.device fails when device_map="auto" splits layers across devices
    input_ids = inputs["input_ids"].to("cuda")
    attention_mask = inputs["attention_mask"].to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Slice off the prompt tokens — only decode the newly generated part
    generated_tokens = outputs[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

# ── Test it ──────────────────────────────────────────────────────────────────
print(predict("What is the difference between L1 and L2 regularization?"))
print("---")
